# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

print(f"Dataset: {metadata['name']}")
print(f"Description: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

> **Note:** All entities (Record Sets, Fields, Columns) are referenced by their `@id`.


In [ ]:
# Get all recordSet @ids from the dataset metadata
record_sets = []
if 'recordSet' in metadata:
    if isinstance(metadata['recordSet'], list):
        record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in metadata['recordSet']]
    elif isinstance(metadata['recordSet'], dict) and '@id' in metadata['recordSet']:
        record_sets = [metadata['recordSet']['@id']]

print(f"Available record sets (@id): {record_sets}")

# Print available fields and columns for each record set
for rs_id in record_sets:
    print(f"\n--- Record Set: {rs_id} ---")
    recset = None
    if 'recordSet' in metadata:
        candidates = metadata['recordSet']
        if isinstance(candidates, dict):
            candidates = [candidates]
        for obj in candidates:
            if ('@id' in obj and obj['@id'] == rs_id) or obj == rs_id:
                recset = obj
                break
    if not recset:
        continue
    if 'field' in recset:
        fields = recset['field']
        if not isinstance(fields, list):
            fields = [fields]
        print('Fields in record set:')
        for field in fields:
            if isinstance(field, dict) and '@id' in field:
                print(f" - {field['@id']}")
            elif isinstance(field, str):
                print(f" - {field}")
    if 'column' in recset:
        columns = recset['column']
        if not isinstance(columns, list):
            columns = [columns]
        print('Columns in record set:')
        for col in columns:
            if isinstance(col, dict) and '@id' in col:
                print(f" - {col['@id']}")
            elif isinstance(col, str):
                print(f" - {col}")


## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field/column `@id`s from the overview.

In [ ]:
# Use the discovered record set @ids to extract data
# Some datasets expose only one record set. List all discovered ones for completeness.

dataframes = {}
for rs_id in record_sets:
    print(f"Loading data for record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
        else:
            df = pd.DataFrame()
        dataframes[rs_id] = df
        print(f"Columns in {rs_id}:", df.columns.tolist())
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# Choose the first record set (if any) as the main table for demonstration
if record_sets:
    main_record_set = record_sets[0]
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes outlier removal, transformation, and grouping records by attributes using field `@id`s.

In [ ]:
# Select numeric and categorical field @ids for analysis. For demonstration, pick likely numeric and group columns (edit by overview section as needed):
main_df = dataframes[main_record_set] if record_sets else pd.DataFrame()

# Show columns to help user select
print(f"Columns in main DataFrame: {main_df.columns.tolist()}")

# Auto-search likely numeric and group fields by simple heuristics
numeric_field_id = None
group_field_id = None
for name in main_df.columns:
    if numeric_field_id is None and main_df[name].dtype in ['int64', 'float64']:
        numeric_field_id = name
    if group_field_id is None and name.lower().startswith(('group', 'type', 'class', 'category')):
        group_field_id = name
if numeric_field_id is None:
    for name in main_df.columns:
        # Try to pick a field likely to be numeric
        if any(sub in name.lower() for sub in ['abundance', 'count', 'quant', 'mw', 'coverage']):
            numeric_field_id = name
            break

# If group field not found, pick the first non-numeric
if group_field_id is None:
    for name in main_df.columns:
        if main_df[name].dtype == 'object':
            group_field_id = name
            break

print(f"Using numeric field (@id): {numeric_field_id}")
print(f"Using group field (@id): {group_field_id}")

# Drop records with missing values in the numeric field (for analysis)
filtered_df = main_df.copy()
if numeric_field_id and numeric_field_id in filtered_df.columns:
    filtered_df = filtered_df.dropna(subset=[numeric_field_id])
    try:
        filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
        # Only keep rows with a valid value
        filtered_df = filtered_df[filtered_df[numeric_field_id].notnull()]
        # Set an arbitrary threshold: e.g., above the 75th percentile
        threshold = filtered_df[numeric_field_id].quantile(0.75)
        filtered_rec = filtered_df[filtered_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_rec.head())

        # Normalize the numeric field
        filtered_rec[f"{numeric_field_id}_normalized"] = (
            (filtered_rec[numeric_field_id] - filtered_rec[numeric_field_id].mean()) / filtered_rec[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_rec[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field_id if exists
        if group_field_id and group_field_id in filtered_rec.columns:
            grouped_df = filtered_rec.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    except Exception as e:
        print(f"Could not process numeric field for EDA: {e}")
else:
    print("No suitable numeric field found in this dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram/boxplot of the numeric field
if numeric_field_id and numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f'Histogram of {numeric_field_id}')

    plt.subplot(1, 2, 2)
    sns.boxplot(x=filtered_df[numeric_field_id])
    plt.title(f'Boxplot of {numeric_field_id}')
    plt.tight_layout()
    plt.show()

# Barplot by group field if exists
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrates how to load, explore, and visualize data from a Croissant-compliant dataset using the `mlcroissant` library. 

Key findings:
- We loaded record sets and fields dynamically using their `@id`s as defined in the schema.
- Performed simple filtering and normalization on a numeric field, and grouped by a category if available.
- Visualized distributions to help inform further proteomics or experimental analysis.

<br/>
Explore additional analysis by applying other processing or machine learning methods to this data using `mlcroissant` and your preferred libraries!
